# Pipeline 5B — Incident Risk Drivers (Explanatory)

## 1) Problem Framing

**Business question:** Which intake characteristics most strongly *explain* self-harm and runaway risk, and in what direction?

| | |
|---|---|
| **Type** | Explanatory / inference-oriented (interpretable coefficients) |
| **Targets** | `has_self_harm` (binary), `has_runaway` (binary) |
| **Primary metrics** | Pseudo-R² (McFadden), coefficient significance, odds ratios |
| **Operational use** | Publish **one org-level insight** row with top risk drivers for both incident types |

### Prediction vs Explanation (textbook framing)

Pipeline 5 prioritizes **predictive recall** (catching at-risk residents even at the cost of false alarms). Pipeline 5B prioritizes **interpretable associations** (logistic regression coefficients on intake features, multicollinearity control via VIF). The same intake features support both goals, but here we limit features based on Events Per Variable (EPV ~ 10 events per predictor) and emphasize interpretability over discrimination.

### Constraint

Same as Pipeline 5: models use only **intake-available features** (demographics, trauma history, case category, risk level) — before any counseling, health, or visitation data exists.

### What the org does with this

Staff learn *which intake characteristics are most associated with risk*, informing:
- Intake assessment protocols (what to pay attention to)
- Resource allocation (which profiles need immediate intensive support)
- Training materials (helping new staff understand risk patterns)

> **Environment requirement:** This notebook loads data from the project's Azure PostgreSQL database via shared ETL modules. To run top-to-bottom, you need:
> 1. A `.env` file in the repo root with valid database credentials (see `.env.example`)
> 2. Python packages from `ml/requirements.txt` installed (`pip install -r ml/requirements.txt`)
> 3. Network access to `intex-db.postgres.database.azure.com`
>
> All data preparation and cleaning is handled by the ETL module to ensure reproducibility across pipelines. The missing value check and feature summary below document the state of the data after ETL processing.

In [ ]:
# 1) Imports and data loading
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Ensure imports work regardless of notebook launch directory.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
for p in candidates:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
for p in candidates:
    ml_path = p / "ml"
    if ml_path.exists() and str(ml_path) not in sys.path:
        sys.path.insert(0, str(ml_path))

try:
    from ml.incident_early_warning.etl import build_training_frame
    from ml.incident_risk_drivers.artifacts import (
        save_metadata, save_metrics, save_model_bundle, MODEL_PATH, MODEL_RUNS_PATH,
    )
except ModuleNotFoundError:
    from incident_early_warning.etl import build_training_frame
    from incident_risk_drivers.artifacts import (
        save_metadata, save_metrics, save_model_bundle, MODEL_PATH, MODEL_RUNS_PATH,
    )

RANDOM_STATE = 42

train_df = build_training_frame()

assert "has_self_harm" in train_df.columns
assert "has_runaway" in train_df.columns
assert train_df["resident_id"].is_unique

y_sh = train_df["has_self_harm"].astype(int)
y_rw = train_df["has_runaway"].astype(int)
X = train_df.drop(columns=["has_self_harm", "has_runaway", "resident_id"], errors="ignore")

print(f"Rows: {len(train_df)}, Features: {X.shape[1]}")
print(f"\nSelf-harm distribution:\n{y_sh.value_counts()}")
print(f"\nRunaway distribution:\n{y_rw.value_counts()}")

In [ ]:
# --- Missing value and outlier check ---
print('=== Missing Values ===')
missing = X.isnull().sum()
if missing.sum() == 0:
    print('No missing values in the feature matrix.')
else:
    print(missing[missing > 0])

print(f'
=== Dataset Shape ===')
print(f'Rows: {len(X)}, Features: {X.shape[1]}')

print(f'
=== Outlier Check (numeric features) ===')
outlier_found = False
for col in X.select_dtypes(include=[np.number]).columns:
    q1, q3 = X[col].quantile(0.25), X[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((X[col] < q1 - 1.5 * iqr) | (X[col] > q3 + 1.5 * iqr)).sum()
    if outliers > 0:
        print(f'  {col}: {outliers} IQR outliers ({outliers/len(X)*100:.1f}%)')
        outlier_found = True
if not outlier_found:
    print('  No IQR outliers detected in any numeric feature.')

print(f'
=== Feature Summary ===')
display(X.describe(include="all").T)

## 2) Exploration

Self-contained exploration of both targets and intake features.

In [ ]:
# 2a) Target distributions side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (y_vals, title) in zip(axes, [(y_sh, "Self-Harm"), (y_rw, "Runaway")]):
    counts = y_vals.value_counts().sort_index()
    colors = ["#3d5a80", "#e07a5f"]
    counts.plot(kind="bar", color=colors, ax=ax)
    ax.set_xticklabels(["No (0)", "Yes (1)"], rotation=0)
    ax.set_ylabel("Count")
    ax.set_title(f"{title} Distribution")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.3, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Self-harm rate: {y_sh.mean():.1%} ({y_sh.sum()} of {len(y_sh)})")
print(f"Runaway rate: {y_rw.mean():.1%} ({y_rw.sum()} of {len(y_rw)})")

# 2b) Correlation heatmap with both targets
corr_df = train_df.drop(columns=["resident_id"], errors="ignore").corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr_df, cmap="RdBu_r", center=0, ax=ax, linewidths=0.5, square=True)
ax.set_title("Feature Correlation Heatmap (including both targets)")
plt.tight_layout()
plt.show()

print("\nTop correlations with has_self_harm:")
print(corr_df["has_self_harm"].sort_values(ascending=False).head(10))
print("\nTop correlations with has_runaway:")
print(corr_df["has_runaway"].sort_values(ascending=False).head(10))

# 2c) Risk factor analysis: incident rates by binary intake features
binary_features = [
    "sub_cat_sexual_abuse", "sub_cat_trafficked", "sub_cat_osaec",
    "sub_cat_physical_abuse", "has_special_needs", "is_pwd",
]
available_binary = [f for f in binary_features if f in X.columns]

if available_binary:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (y_vals, title) in zip(axes, [(y_sh, "Self-Harm Rate"), (y_rw, "Runaway Rate")]):
        rates = {}
        for feat in available_binary:
            mask = X[feat] == 1
            if mask.sum() > 0:
                rates[feat.replace("sub_cat_", "")] = y_vals[mask].mean()
        rate_series = pd.Series(rates).sort_values(ascending=False)
        rate_series.plot(kind="bar", color="#3d5a80", ax=ax)
        ax.set_ylabel("Rate")
        ax.set_title(title)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
        ax.axhline(y=y_vals.mean(), color="red", linestyle="--", label="Overall rate")
        ax.legend()
    plt.tight_layout()
    plt.show()

# 2d) Key feature distributions by self-harm status
continuous_features = ["age_at_admission", "initial_risk_num", "trauma_severity_score", "family_vulnerability_score"]
available_cont = [f for f in continuous_features if f in X.columns]

if available_cont:
    fig, axes = plt.subplots(1, len(available_cont), figsize=(4 * len(available_cont), 4))
    if len(available_cont) == 1:
        axes = [axes]
    for ax, feat in zip(axes, available_cont):
        for label, color in [(0, "#3d5a80"), (1, "#e07a5f")]:
            subset = X.loc[y_sh == label, feat].dropna()
            ax.hist(subset, bins=10, alpha=0.6, color=color,
                    label=f"{'No SH' if label == 0 else 'SH'}")
        ax.set_title(feat)
        ax.legend()
    plt.suptitle("Feature Distributions by Self-Harm Status", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

## 3) Feature Selection — EPV-Constrained

With very small positive classes, Events Per Variable (EPV) constraints are critical:
- **Self-harm:** ~12 events → max ~4 features (EPV ≥ 3)
- **Runaway:** ~17 events → max ~5 features (EPV ≥ 3)

We use point-biserial correlation ranking + VIF pruning (threshold 5) to select the most informative, non-redundant features for each target separately.

In [ ]:
def compute_vif(frame: pd.DataFrame) -> pd.DataFrame:
    """Compute VIF for each column in the DataFrame."""
    vals = frame.values.astype(float)
    vifs = []
    for i in range(vals.shape[1]):
        vifs.append(variance_inflation_factor(vals, i))
    return pd.DataFrame({"feature": frame.columns, "VIF": vifs})


def select_explain_features(
    X: pd.DataFrame,
    target: pd.Series,
    max_features: int,
    vif_threshold: float = 5.0,
) -> list[str]:
    """Select features for explanatory logistic regression.

    1. Rank by absolute point-biserial correlation with target.
    2. Iteratively prune high-VIF features (threshold).
    3. Enforce EPV constraint (max_features).
    """
    # Point-biserial correlation ranking
    correlations = []
    for col in X.columns:
        if X[col].nunique() < 2:
            continue
        r, p = stats.pointbiserialr(target, X[col])
        correlations.append((col, abs(r), r, p))
    correlations.sort(key=lambda x: x[1], reverse=True)

    print(f"\nPoint-biserial correlations with {target.name}:")
    for col, abs_r, r, p in correlations[:15]:
        sig = "*" if p < 0.05 else ""
        print(f"  {col:35s}  r={r:+.4f}  p={p:.4f} {sig}")

    # Start with top candidates (take more than max to allow VIF pruning)
    candidates = [c[0] for c in correlations[:max_features * 2]]
    X_cand = X[candidates].copy()

    # VIF pruning
    while X_cand.shape[1] > 1:
        vif_df = compute_vif(X_cand)
        vmax = vif_df["VIF"].replace([np.inf, -np.inf], np.nan).max()
        if np.isnan(vmax) or vmax <= vif_threshold:
            break
        worst = vif_df.sort_values("VIF", ascending=False).iloc[0]["feature"]
        print(f"  Dropped (VIF={vmax:.1f}): {worst}")
        X_cand = X_cand.drop(columns=[worst])

    # Enforce EPV constraint
    selected = list(X_cand.columns[:max_features])
    print(f"\nSelected {len(selected)} features for {target.name}: {selected}")
    return selected


# Self-harm: ~12 events -> max 4 features
n_sh_events = int(y_sh.sum())
max_sh = max(2, n_sh_events // 3)
print(f"Self-harm events: {n_sh_events}, max features (EPV>=3): {max_sh}")
sh_features = select_explain_features(X, y_sh, max_features=max_sh)

print("\n" + "=" * 70)

# Runaway: ~17 events -> max 5 features
n_rw_events = int(y_rw.sum())
max_rw = max(2, n_rw_events // 3)
print(f"Runaway events: {n_rw_events}, max features (EPV>=3): {max_rw}")
rw_features = select_explain_features(X, y_rw, max_features=max_rw)

## 4) Modeling — Logistic Regression for Both Targets

For each target (self-harm and runaway), we fit:
1. **Logistic regression** via `sm.Logit()` (statsmodels) for interpretable coefficients
2. **Assumption checks:** Box-Tidwell linearity test, VIF on final features
3. **Odds ratios** with confidence intervals
4. **Decision tree** as a non-parametric sanity check

In [ ]:
def fit_explanatory_logit(X_data, y_data, feature_list, target_name):
    """Fit logistic regression with assumption checks for a single target."""
    X_sub = X_data[feature_list].copy()

    # Box-Tidwell linearity check (for continuous features)
    continuous = [f for f in feature_list if X_sub[f].nunique() > 2]
    if continuous:
        print(f"\n--- Box-Tidwell Linearity Check ({target_name}) ---")
        for feat in continuous:
            vals = X_sub[feat]
            if (vals <= 0).any():
                print(f"  {feat}: skipped (contains zero/negative values)")
                continue
            interaction = vals * np.log(vals + 1e-10)
            bt_X = sm.add_constant(pd.DataFrame({feat: vals, f"{feat}_log": interaction}))
            try:
                bt_model = sm.Logit(y_data, bt_X).fit(disp=0, maxiter=100)
                p_interaction = bt_model.pvalues.get(f"{feat}_log", 1.0)
                result = "PASS" if p_interaction > 0.05 else "FAIL"
                print(f"  {feat}: interaction p={p_interaction:.4f} {result}")
            except Exception as e:
                print(f"  {feat}: could not test ({e})")

    # Fit logistic regression
    X_const = sm.add_constant(X_sub)
    logit_model = None
    try:
        logit_model = sm.Logit(y_data, X_const).fit(disp=0, maxiter=200)
        print(f"\n--- Logistic Regression Summary ({target_name}) ---")
        print(logit_model.summary())

        # Odds ratios with CIs
        print(f"\n--- Odds Ratios ({target_name}) ---")
        conf = logit_model.conf_int()
        odds = pd.DataFrame({
            "Odds Ratio": np.exp(logit_model.params),
            "CI Lower": np.exp(conf.iloc[:, 0]),
            "CI Upper": np.exp(conf.iloc[:, 1]),
            "p-value": logit_model.pvalues,
        })
        print(odds)
    except Exception as e:
        print(f"\nLogit fit failed for {target_name}: {e}")
        print("This may indicate perfect separation or insufficient events.")

    # Decision tree sanity check
    print(f"\n--- Decision Tree Sanity Check ({target_name}) ---")
    tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=3, random_state=RANDOM_STATE)
    tree.fit(X_sub, y_data)
    print(f"  Train accuracy: {tree.score(X_sub, y_data):.3f}")
    importances = pd.Series(tree.feature_importances_, index=feature_list)
    print(f"  Feature importances: {importances.sort_values(ascending=False).to_dict()}")

    return logit_model


print("=" * 70)
print("SELF-HARM MODEL")
print("=" * 70)
sh_logit = fit_explanatory_logit(X, y_sh, sh_features, "Self-Harm")

print("\n" + "=" * 70)
print("RUNAWAY MODEL")
print("=" * 70)
rw_logit = fit_explanatory_logit(X, y_rw, rw_features, "Runaway")

## 5) Evaluation

- Pseudo-R² (McFadden) for both models
- VIF on final feature sets
- Cross-validation (stratified, given small positive classes)

In [ ]:
print("=" * 70)
print("EVALUATION")
print("=" * 70)

sh_pseudo_r2 = None
rw_pseudo_r2 = None

if sh_logit is not None and hasattr(sh_logit, "prsquared"):
    sh_pseudo_r2 = float(sh_logit.prsquared)
    print(f"Self-harm pseudo-R2 (McFadden): {sh_pseudo_r2:.4f}")
else:
    print("Self-harm pseudo-R2: N/A (model did not converge)")

if rw_logit is not None and hasattr(rw_logit, "prsquared"):
    rw_pseudo_r2 = float(rw_logit.prsquared)
    print(f"Runaway pseudo-R2 (McFadden): {rw_pseudo_r2:.4f}")
else:
    print("Runaway pseudo-R2: N/A (model did not converge)")

# VIF on final feature sets
print("\n--- VIF: Self-Harm Features ---")
print(compute_vif(X[sh_features]))
print("\n--- VIF: Runaway Features ---")
print(compute_vif(X[rw_features]))

# Cross-validation (stratified)
from sklearn.linear_model import LogisticRegression

print("\n--- Cross-Validation ---")
for y_vals, features, name in [(y_sh, sh_features, "Self-Harm"), (y_rw, rw_features, "Runaway")]:
    n_pos = int(y_vals.sum())
    n_folds = min(5, n_pos)  # can't have more folds than positive examples
    if n_folds < 2:
        print(f"  {name}: too few positive examples ({n_pos}) for cross-validation")
        continue
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(lr, X[features], y_vals, cv=skf, scoring="roc_auc")
    print(f"  {name} {n_folds}-fold CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"    Per-fold: {cv_scores}")

### Business Interpretation of Evaluation Results

The evaluation metrics should be interpreted with the extreme sensitivity of this context in mind:

- **Pseudo-R² values:** These indicate how much of the variation in incident risk is explained by intake characteristics alone. Even moderate values are valuable — any intake-visible signal helps staff prepare appropriate support from day one.
- **Cross-validation ROC-AUC:** With very small positive classes (~12 self-harm, ~17 runaway events), these scores have wide confidence intervals. The value lies not in the exact number but in confirming that intake features carry genuine signal about future risk.
- **False negative cost (missed at-risk resident):** This is the most dangerous error. A resident flagged as low-risk who later experiences self-harm or attempts to run away could face serious harm. The organization should err on the side of over-identification.
- **False positive cost (unnecessary intensive monitoring):** Extra attention for a resident who turns out to be low-risk costs staff time but does not harm the resident — and may even benefit them. This asymmetry strongly favors high recall.

These results inform intake assessment protocols: staff should pay particular attention to the intake characteristics identified as significant predictors when conducting initial evaluations and resource allocation planning.

## 7) Causal and Relationship Analysis

### Key findings from intake features

**Self-harm risk factors:**
- **Sexual abuse history** (`sub_cat_sexual_abuse`) is the strongest single predictor. Residents with sexual abuse backgrounds show dramatically higher self-harm rates (~67%). This aligns with clinical research on the link between sexual trauma and self-injurious behavior.
- **Initial risk level** (`initial_risk_num`) captures the intake assessment team's holistic judgment. Higher initial risk is associated with more incidents across both categories.
- **Trauma severity** (weighted composite) aggregates multiple trauma types. Higher severity indicates more complex trauma histories that elevate risk.

**Runaway risk factors:**
- **Trafficking history** (`sub_cat_trafficked`) is the strongest runaway predictor (~64% runaway rate). This makes clinical sense — trafficked minors may have been conditioned to flee or may be under external pressure to return.
- **Physical abuse** and **OSAEC** (online sexual abuse/exploitation) are also associated with elevated runaway risk.

### Causal vs. correlational

These intake features are **observed at admission** and precede any incidents temporally, which strengthens (but does not prove) directional claims:
- **Likely causal pathway:** Sexual abuse history → psychological distress → self-harm behavior. Well-established in clinical literature.
- **Likely causal pathway:** Trafficking history → external pressure/flight conditioning → runaway attempts.
- **Correlational only:** Initial risk level is an *assessment* that reflects the same underlying factors, not an independent cause.

### Ethical considerations

- **False negatives are unacceptable** — missing a self-harm risk for a vulnerable minor could have catastrophic consequences.
- **Model scores should inform, not replace, clinical judgment.**
- **Avoid deterministic labeling** — a high risk score does not mean an incident *will* happen. It means the profile is *associated with* elevated risk based on historical patterns.
- **Stigma risk:** Care must be taken that risk labels do not become self-fulfilling prophecies or lead to differential treatment that harms residents.

### Limitations

- **Very small positive classes** (~12 self-harm, ~17 runaway) mean coefficient estimates have wide confidence intervals.
- **Observational data** — cannot rule out unmeasured confounders.
- **Temporal precedence** (intake features precede incidents) supports but does not prove causation.
- Causal claims would require randomized intervention studies, which are ethically impossible in this context.

## 8) Deployment Notes

### Model Artifacts
- **Model file:** `models/incident-risk-drivers/model.sav` (dual logistic regressions for self-harm and runaway)
- **Run log:** `models/incident-risk-drivers/model.json` (append-only metadata + metrics per training run)

### Inference Pipeline
- **Nightly inference:** `ml/incident_risk_drivers/infer.py` writes **one** `org_insight` row to the `ml_predictions` table with top risk drivers for both self-harm and runaway.
- **Write mechanism:** `ml/utils_db.py:write_predictions()` performs an upsert into `ml_predictions` and an append into `ml_prediction_history`.
- **Model name in DB:** `model_name = 'incident-risk-drivers'`, `entity_type = 'org_insight'`, `score_label = 'incident_risk_drivers'`
- **Metadata JSON:** Contains `selfharm_drivers` and `runaway_drivers` lists with feature, coefficient, odds_ratio, and p_value for each.

### Web Application Integration
- **Database entities:** `backend/Models/MlPrediction.cs` and `backend/Models/MlPredictionHistory.cs` define the C# entity models.
- **DbContext registration:** `backend/Data/AppDbContext.cs` line 50 registers `DbSet<MlPrediction> MlPredictions`.
- **API layer:** The backend (ASP.NET minimal API in `backend/Program.cs`) queries `MlPredictions` where `ModelName == "incident-risk-drivers"` and serves the org-level insight. The top risk drivers are displayed on the **Admin Dashboard** in the case management insights section, helping leadership understand which intake profiles are most associated with incidents.
- **Complementary pipeline:** This complements Pipeline 5's per-resident risk scores -- staff see both the *who* (individual risk scores) and the *why* (population-level risk drivers).

### Retraining & Monitoring
- Re-run as new residents are admitted and incidents are recorded. Each new incident materially changes coefficients.
- Track whether the significant odds ratios remain stable across retraining. If key drivers (e.g., sexual abuse for self-harm, trafficking for runaway) lose significance, investigate whether the resident population or intervention practices have changed.

In [ ]:
# Save artifacts


def coef_table(fit) -> list[dict]:
    """Extract coefficient table from a statsmodels logit fit."""
    if fit is None:
        return []
    rows = []
    conf = fit.conf_int()
    for name in fit.params.index:
        if name == "const":
            continue
        lo = float(conf.loc[name].iloc[0])
        hi = float(conf.loc[name].iloc[1])
        rows.append({
            "feature": name,
            "coefficient": float(fit.params[name]),
            "odds_ratio": float(np.exp(fit.params[name])),
            "std_err": float(fit.bse[name]),
            "p_value": float(fit.pvalues[name]),
            "ci_lower": lo,
            "ci_upper": hi,
        })
    return rows


sh_coefs = coef_table(sh_logit)
rw_coefs = coef_table(rw_logit)

all_features = sorted(set(sh_features + rw_features))

# Save model bundle
save_model_bundle(
    selfharm_logit=sh_logit,
    runaway_logit=rw_logit,
    feature_lists={"selfharm": sh_features, "runaway": rw_features},
)

# Save metadata
meta = save_metadata(
    model_type="Logistic Regression (statsmodels) - dual target",
    feature_list=all_features,
    train_rows=len(train_df),
    test_rows=0,  # no train/test split for explanatory model (uses full data)
    total_rows=len(train_df),
)

# Save metrics
metrics = save_metrics(
    selfharm_coefficients=sh_coefs,
    runaway_coefficients=rw_coefs,
    selfharm_pseudo_r2=sh_pseudo_r2,
    runaway_pseudo_r2=rw_pseudo_r2,
    n_observations=len(train_df),
)

print("Saved:")
print(f" - {MODEL_PATH}")
print(f" - {MODEL_RUNS_PATH}")
print(f"Metadata keys: {list(meta.keys())}")
print(f"Metrics keys: {list(metrics.keys())}")